In [0]:
dbutils.widgets.text("catalog","claims")
catalog = dbutils.widgets.get("catalog")

In [0]:
df = spark.table(f"{catalog}.silver.claims_clean")

In [0]:
from pyspark.sql.functions import *

claims_src = (spark.read.table(f"{catalog}.silver.claims_clean")
    # Silver type drifts with bronze (bronze stores everything as STRING) — normalize once here
    .withColumn("admission_date", col("admission_date").cast("date"))
    .withColumn("discharge_date", col("discharge_date").cast("date"))
    .withColumn("submitted_ts", col("submitted_ts").cast("timestamp"))
    .withColumn("settled_ts", col("settled_ts").cast("timestamp"))
    .withColumn("billed_amount", col("billed_amount").cast("double"))
    .withColumn("approved_amount", col("approved_amount").cast("double"))
    .withColumn("room_days", col("room_days").cast("long"))
    .withColumn("room_rent_per_day", col("room_rent_per_day").cast("long")))
dim_policy = spark.read.table(f"{catalog}.silver.dim_policy_scd2")
plans = spark.read.table(f"{catalog}.raw.plans")
hospitals = spark.read.table(f"{catalog}.raw.hospitals")

# As-of plan lookup: the dim_policy_scd2 version valid on admission_date
plan_as_of_admission = (claims_src
    .select("policy_id", "admission_date").distinct().alias("c")
    .join(dim_policy.alias("d"),
          (col("c.policy_id") == col("d.policy_id")) &
          (col("d.effective_from") <= col("c.admission_date")) &
          (col("d.end_date").isNull() | (col("c.admission_date") < col("d.end_date"))),
          "left")
    .select(
        col("c.policy_id"),
        col("c.admission_date"),
        col("d.plan_code").alias("plan_code_at_admission")))

# Policy inception: earliest effective date across the policy's full history
policy_inception = (dim_policy
    .groupBy("policy_id")
    .agg(min("effective_from").alias("policy_start_date")))

network_history = (spark.read.format("csv")
    .option("header", "true")
    .schema("hospital_id STRING, network_from DATE, network_to DATE")
    .load("/Volumes/claims/raw/landing/hospital_network_history.csv"))

enriched = (claims_src
    .join(plan_as_of_admission, ["policy_id", "admission_date"], "left")
    .join(policy_inception, "policy_id", "left")
    .join(plans.select(
            col("plan_code").alias("plan_code_at_admission"),
            col("plan_family"),
            col("room_rent_cap_per_day")),
          "plan_code_at_admission", "left")
    .join(hospitals.select("hospital_id", "city_tier"), "hospital_id", "left"))

network_flags = (claims_src.select("claim_id", "hospital_id", "admission_date")
    .join(network_history, "hospital_id", "left")
    .withColumn("in_network_on_date",
        col("network_from").isNotNull() &
        (col("network_from") <= col("admission_date")) &
        (col("network_to").isNull() | (col("admission_date") <= col("network_to"))))
    .groupBy("claim_id")
    .agg(max("in_network_on_date").alias("is_network_at_admission")))

# R5 deliverable: exactly 30 columns, no more, no fewer
claims_enriched_df = (enriched
    .join(network_flags, "claim_id", "left")
    .select(
        col("claim_id"),
        col("member_id"),
        col("policy_id"),
        col("hospital_id"),
        col("admission_date"),
        col("discharge_date"),
        date_trunc("month", col("admission_date")).cast("date").alias("admission_month"),
        col("diagnosis_code"),
        col("procedure_code"),
        col("is_pre_existing"),
        col("claim_type"),
        col("billed_amount"),
        col("room_rent_per_day"),
        col("room_days"),
        col("status"),
        col("approved_amount"),
        col("rejection_code"),
        col("submitted_ts"),
        col("settled_ts"),
        col("channel"),
        col("plan_code_at_admission"),
        col("plan_family"),
        col("room_rent_cap_per_day"),
        col("policy_start_date"),
        floor(months_between(col("admission_date"), col("policy_start_date"))).cast("int").alias("policy_tenure_months"),
        datediff(col("discharge_date"), col("admission_date")).cast("int").alias("length_of_stay"),
        (col("room_rent_per_day") > col("room_rent_cap_per_day")).alias("room_rent_over_cap"),
        when(col("settled_ts").isNotNull(),
             datediff(col("settled_ts"), col("submitted_ts")).cast("int")).alias("turnaround_days"),
        coalesce(col("is_network_at_admission"), lit(False)).alias("is_network_at_admission"),
        col("city_tier").alias("hospital_city_tier"),
    ))

# R7: claims_enriched must be clustered by admission_month, hospital_id.
# Always overwrite so gold reflects the current silver snapshot (create-if-absent left a stale table).
(claims_enriched_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .clusterBy("admission_month", "hospital_id")
    .saveAsTable(f"{catalog}.gold.claims_enriched"))

In [0]:
claims_enriched_df.count()

In [0]:
claims_gold = spark.read.table(f"{catalog}.gold.claims_enriched")

hospital_monthly_df = (claims_gold
    .groupBy("hospital_id", "admission_month")
    .agg(
        count("*").cast("int").alias("claim_count"),
        round(sum("billed_amount"), 2).cast("decimal(18,2)").alias("total_billed"),
        coalesce(round(sum("approved_amount"), 2), lit(0.0))
            .cast("decimal(18,2)").alias("total_approved"),
        sum(when(col("status") == "rejected", 1).otherwise(0))
            .cast("int").alias("rejected_count")))

# Idempotent publish: overwrite on every run (claims_daily job runs this daily).
(hospital_monthly_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(f"{catalog}.gold.hospital_monthly"))

# R2 documentation is schema-level metadata; re-apply it after every overwrite
spark.sql(f"COMMENT ON TABLE {catalog}.gold.hospital_monthly IS 'Gold monthly aggregate per hospital and admission month: claim counts, billed/approved totals and rejected counts.'")
monthly_comments = {
    "hospital_id": "Hospital identifier",
    "admission_month": "First-of-month bucket of admission_date",
    "claim_count": "Number of claims for the hospital-month",
    "total_billed": "Sum of billed_amount for the hospital-month",
    "total_approved": "Sum of approved_amount for the hospital-month, coalesced to 0 when none approved",
    "rejected_count": "Number of claims with status = rejected",
}
for mcol, cmt in monthly_comments.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.hospital_monthly ALTER COLUMN {mcol} COMMENT '{cmt}'")

display(spark.sql(f"SELECT count(*) AS hospital_monthly_rows FROM {catalog}.gold.hospital_monthly"))

In [0]:
# from pyspark.sql.functions import *

# # R5: monthly per-hospital claims summary with exact target types:
# # hospital_id, admission_month DATE, claim_count INT,
# # total_billed DECIMAL(14,2), total_approved DECIMAL(14,2), rejected_count INT
# hospital_monthly_df = (spark.table(f"{catalog}.gold.claims_enriched")
#     .groupBy("hospital_id", "admission_month")
#     .agg(
#         count("*").cast("int").alias("claim_count"),
#         sum("billed_amount").cast("decimal(14,2)").alias("total_billed"),
#         coalesce(sum("approved_amount"), lit(0.0)).cast("decimal(14,2)").alias("total_approved"),
#         sum(when(col("status") == "rejected", lit(1)).otherwise(lit(0))).cast("int").alias("rejected_count"))
#     .select("hospital_id", "admission_month", "claim_count",
#             "total_billed", "total_approved", "rejected_count"))

# (hospital_monthly_df.write
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(f"{catalog}.gold.hospital_monthly"))

# spark.table(f"{catalog}.gold.hospital_monthly").printSchema()
# display(spark.table(f"{catalog}.gold.hospital_monthly").orderBy("hospital_id", "admission_month").limit(10))